# 🤖 Task 5: Auto Tagging Support Tickets Using LLM

### Project Overview
Customer service teams handle a massive volume of daily queries. Manually reading and routing these tickets is highly inefficient.
This notebook demonstrates an automated NLP pipeline that leverages a Large Language Model (LLM) to read customer queries and instantly predict the **top 3 most applicable categories**.

### Dataset Used
We utilize the **Banking77** dataset from Hugging Face, which contains various banking-related customer service requests (e.g., card issues, top-up failures, etc.).

### Methodology
We will implement and compare two prompting strategies using a pre-trained Transformer model:
1. **Zero-Shot Classification:** Categorizing tickets purely based on label semantics without prior examples.
2. **Few-Shot Learning:** Injecting context by providing the model with a few labeled examples before asking it to classify a new ticket.

In [ ]:
# Import necessary libraries for data manipulation and modeling
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from transformers import pipeline
from tqdm import tqdm

### 1. Data Acquisition
Fetching the Banking77 dataset via the Hugging Face Hub.

In [ ]:
# Load the dataset
hf_dataset = load_dataset("banking77")
hf_dataset

### 2. Data Preprocessing
Converting the raw dataset into a Pandas DataFrame for easier inspection and manipulation.

In [ ]:
# Convert to DataFrame
support_df = pd.DataFrame(hf_dataset["train"])
support_df.head()

### Feature Selection
For this pipeline, we only need the customer's text. We will isolate it and rename the column to represent a support ticket.

In [ ]:
# Keep only the text column and rename it for clarity
support_df = support_df[["text"]]
support_df = support_df.rename(columns={"text": "customer_query"})
support_df.head()

# Check original dimensions
support_df.shape

In [ ]:
# Check original dimensions
support_df.shape

### Downsampling for Prototyping
To speed up the inference time during this experiment, we will take a random sample of 100 tickets.

In [ ]:
# Sample 100 records
support_df = support_df.sample(100, random_state=42)
support_df.head()

### 3. Defining the Target Categories
Here we define the predefined tags that the support system uses to route tickets.

In [ ]:
# Define the business categories
ticket_categories = [
    "Account Access Problem",
    "Payment or Billing Issue",
    "Card Problem",
    "Transaction Issue",
    "Technical App Problem",
    "Refund Request",
    "Fraud or Security Issue",
    "General Inquiry"
]
ticket_categories

### 4. Zero-Shot Classification Pipeline
We instantiate a pre-trained sequence classifier.
*(Note: Zero-shot requires no fine-tuning; the model relies on its pre-existing language understanding to map text to the provided labels).*

In [ ]:
# Initialize the zero-shot classifier directly from Hugging Face Hub
zs_classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli"
)

### Testing Zero-Shot on a Single Query

In [ ]:
# Extract one query to test
sample_query = support_df["customer_query"].iloc[0]

# Run prediction
zs_result = zs_classifier(sample_query, ticket_categories)
zs_result

### Isolating Top 3 Predictions

In [ ]:
# Slice the top 3 highest probability labels
top_three_preds = zs_result["labels"][:3]
top_three_preds

### Executing Zero-Shot Across the Dataset

In [ ]:
zero_shot_output = []

# Iterate through all customer queries
for query in support_df["customer_query"]:
    prediction = zs_classifier(query, ticket_categories)
    top_3_tags = prediction["labels"][:3]
    zero_shot_output.append(top_3_tags)

# Attach results to the dataframe
support_df["zero_shot_preds"] = zero_shot_output
support_df.head()

### 5. Few-Shot Learning Implementation
To enhance accuracy, we will construct a prompt that includes a few labeled examples. This acts as a guide for the LLM to understand the mapping logic better.

In [ ]:
# Define context examples
shot_examples = """
Ticket: I forgot my password and cannot login
Tag: Account Access Problem

Ticket: My card payment failed
Tag: Card Problem

Ticket: I was charged twice for my order
Tag: Payment or Billing Issue
"""

### Building the Prompt Generator

In [ ]:
def generate_few_shot_prompt(query_text):
    prompt_template = f"""
    You are an intelligent support ticket routing assistant.

    Allowed Categories:
    {ticket_categories}

    Reference Examples:
    {shot_examples}

    Task: Analyze the following user query and output the top 3 matching categories.

    Ticket: {query_text}
    """
    return prompt_template

### Validating the Prompt Structure

In [ ]:
# Test the prompt format
sample_query = support_df["customer_query"].iloc[0]
generated_prompt = generate_few_shot_prompt(sample_query)
print(generated_prompt)

### Executing Few-Shot Across the Dataset

In [ ]:
few_shot_output = []

for query in support_df["customer_query"]:
    # Generate the contextual prompt
    structured_prompt = generate_few_shot_prompt(query)

    # Run classification
    prediction = zs_classifier(structured_prompt, ticket_categories)

    # Extract top 3
    top_3_tags = prediction["labels"][:3]
    few_shot_output.append(top_3_tags)

# Attach few-shot results to the dataframe
support_df["few_shot_preds"] = few_shot_output
support_df.head()

### 6. Strategy Comparison
Reviewing the predictions to see how the injected context (Few-Shot) altered the model's routing decisions compared to the baseline (Zero-Shot).

In [ ]:
# Create a cleaner view for comparison
final_comparison = support_df[["customer_query", "zero_shot_preds", "few_shot_preds"]]
final_comparison.head(10)

### 7. Project Conclusion
In this pipeline, we successfully engineered an automated ticketing system using an LLM.

By experimenting with the Banking77 dataset, we demonstrated that:
1. **Zero-Shot Learning** provides a strong, immediate baseline for categorizing text without requiring massive labeled datasets.
2. **Few-Shot Prompting** effectively steers the model's focus, helping it recognize subtle contexts (like distinguishing between a generic card issue and a specific transaction issue) by supplying just a few reference examples.

This implementation highlights the immense potential of LLMs in optimizing customer service operations, reducing manual triage time, and ensuring faster response rates.